In [10]:
import arcpy
import os

In [11]:
# ── PATHS ──────────────────────────────────────────────────────────────────────────────────────────────────

# Input LF and HF interpolated backgroud/mission path data CSVs
lf_csv = r"C:\Users\kkeii\OneDrive\Documents\THESIS_REPOSITORIES\MISSION_REPOSITORIES\UVI2_Around_STX_20250306\data\4_data_interpolated\LF_background_INTERPOLATED_UVI2_Around_STX_20250306.csv"
hf_csv = r"C:\Users\kkeii\OneDrive\Documents\THESIS_REPOSITORIES\MISSION_REPOSITORIES\UVI2_Around_STX_20250306\data\4_data_interpolated\HF_background_INTERPOLATED_UVI2_Around_STX_20250306.csv"


# intermediate point feature classes output by 'XY Table to Point' tool before reprojection (*output to misc_around_stx_2025.gdb)
lf_deploy_g4326 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\LF_deploy_pt_g4326"
hf_deploy_g4326 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\HF_deploy_pt_g4326"
lf_recover_g4326 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\LF_recover_pt_g4326"
hf_recover_g4326 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\HF_recover_pt_g4326"


# Output DEPLOYMENT LOCATION points - glider icon (reprojected to EPSG:26920, *output to GIS_UVI2_Around_STX_20250306.gdb)
lf_deploy_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\LF_deploy_pt_p26920"
hf_deploy_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\HF_deploy_pt_p26920"

# Output RECOVERY LOCATION points - flag icon (reprojected to EPSG:26920, *output to GIS_UVI2_Around_STX_20250306.gdb)
lf_recover_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\LF_recover_pt_p26920"
hf_recover_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\HF_recover_pt_p26920"

In [12]:
# ── SETTINGS ───────────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True

In [13]:
# ── 1. CREATE MISSION START/END POINT FEATURE CLASS FROM CSV DATA ──────────────────────────────────────────

# XY Table To Point (Data Management Tools): Function
def xy_table_to_point(in_table, out_feature_class, x_field, y_field):
    try:
        arcpy.management.XYTableToPoint(
            in_table          = in_table,
            out_feature_class = out_feature_class,
            x_field           = x_field,
            y_field           = y_field,
            z_field           = None,
            # output point feature class in EPSG:4326 (native coordinate system of glider data that background coordinates are interpolated from)
            coordinate_system = arcpy.SpatialReference(4326)
        )
        print(f"✅ Created: {out_feature_class} \n")
    # print warning message if tool fails
    except Exception as e:
        print(f"❌ [FAILURE] XY Table To Point failed for {os.path.basename(out_feature_class)}: {e} \n")

# call function for LF and HF deployment points (use start coordinates)
xy_table_to_point(lf_csv, lf_deploy_g4326, "start_longitude_DD", "start_latitude_DD")
xy_table_to_point(hf_csv, hf_deploy_g4326, "start_longitude_DD", "start_latitude_DD")

# call function for LF and HF recovery points (use end coordinates)
xy_table_to_point(lf_csv, lf_recover_g4326, "end_longitude_DD", "end_latitude_DD")
xy_table_to_point(hf_csv, hf_recover_g4326, "end_longitude_DD", "end_latitude_DD")

✅ Created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\LF_deploy_pt_g4326 

✅ Created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\HF_deploy_pt_g4326 

✅ Created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\LF_recover_pt_g4326 

✅ Created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\HF_recover_pt_g4326 



In [14]:
# ── REPROJECT MISSION POINTS TO EPSG:26920 ──────────────────────────────────────────────────────────────

# Project (Data Management Tools): Function
def project_point_fc(in_dataset, out_dataset):
    try:
        arcpy.management.Project(
            in_dataset              = in_dataset,
            out_dataset             = out_dataset,
            out_coor_system         = arcpy.SpatialReference(26920),
            transform_method        = "WGS_1984_(ITRF00)_To_NAD_1983",
            in_coor_system          = arcpy.SpatialReference(4326),
            # not applicable for point layers (adds extra vertices to lines/polygons)
            preserve_shape          = "NO_PRESERVE_SHAPE"
        )
        print(f"✅ Reprojected: {os.path.basename(out_dataset)}")
    # print warning message if tool fails
    except Exception as e:
        print(f"\n❌ [FAILURE] Reproject failed for {os.path.basename(out_dataset)}: {e} \n")

# call function for each point feature class (input, output)
project_point_fc(lf_deploy_g4326, lf_deploy_p26920)
project_point_fc(hf_deploy_g4326, hf_deploy_p26920)
project_point_fc(lf_recover_g4326,lf_recover_p26920)
project_point_fc(hf_recover_g4326, hf_recover_p26920)

✅ Reprojected: LF_deploy_pt_p26920
✅ Reprojected: HF_deploy_pt_p26920
✅ Reprojected: LF_recover_pt_p26920
✅ Reprojected: HF_recover_pt_p26920


In [15]:
# ── 2a. DEPLOYMENT POINTS: REMOVE ALL ROWS IN ATTRIBUTE TABLE *EXCEPT* STARTING POINT ───────────────────

# Function to find row in Attribute Table with earliest datetime, delete all other rows, and print messages with earliest datetime found + number rows in Attribute Table after removal
def keep_earliest_point(in_fc):
    # print header identifying which feature class this block reports on
    print(f"\n------------- {os.path.basename(in_fc)} -------------")

    # open read-only cursor on datetime field
    with arcpy.da.SearchCursor(in_fc, ["start_datetime_UTC"]) as cur:
        # scan all rows and grab minimum (earliest) datetime
        earliest = min(row[0] for row in cur)
    # print earliest datetime found
    print(f"  ⏱️ Earliest start_datetime_UTC: {earliest}")

    # open editable cursor on datetime field
    with arcpy.da.UpdateCursor(in_fc, ["start_datetime_UTC"]) as cur:
        # track whether earliest row has already been kept
        kept = False
        # step through every row
        for row in cur:
            # keep first row matching the earliest datetime
            if row[0] == earliest and not kept:
                # flag row so no further matches are kept
                kept = True
            else:
                # delete every other row
                cur.deleteRow()
    # report how many rows are left (should be 1)
    print(f"  📑 Rows remaining: {arcpy.management.GetCount(in_fc)[0]}")


# call function for both deployment point feature classes
keep_earliest_point(lf_deploy_p26920)
keep_earliest_point(hf_deploy_p26920)


------------- LF_deploy_pt_p26920 -------------
  ⏱️ Earliest start_datetime_UTC: 2025-03-06 19:23:46+00:00
  📑 Rows remaining: 1

------------- HF_deploy_pt_p26920 -------------
  ⏱️ Earliest start_datetime_UTC: 2025-03-06 19:22:44+00:00
  📑 Rows remaining: 1


In [16]:
# ── 2b. RECOVERY POINTS: REMOVE ALL ROWS IN ATTRIBUTE TABLE *EXCEPT* ENDING POINT ───────────────────────

# Function to find row in Attribute Table with latest datetime, delete all other rows, and print messages with latest datetime found + number rows in Attribute Table after removal
def keep_latest_point(in_fc):
    # print header identifying which feature class this block reports on
    print(f"\n------------- {os.path.basename(in_fc)} -------------")

    # open read-only cursor on datetime field
    with arcpy.da.SearchCursor(in_fc, ["end_datetime_UTC"]) as cur:
        # scan all rows and grab maximum (latest) datetime
        latest = max(row[0] for row in cur)
    # print latest datetime found
    print(f"  ⏱️ Latest end_datetime_UTC: {latest}")

    # open editable cursor on datetime field
    with arcpy.da.UpdateCursor(in_fc, ["end_datetime_UTC"]) as cur:
        # track whether latest row has already been kept
        kept = False
        # step through every row
        for row in cur:
            # keep first row matching the latest datetime
            if row[0] == latest and not kept:
                # flag row so no further matches are kept
                kept = True
            else:
                # delete every other row
                cur.deleteRow()
    # report how many rows are left (should be 1)
    print(f"  📑 Rows remaining: {arcpy.management.GetCount(in_fc)[0]}")


# call function for both recovery point feature classes
keep_latest_point(lf_recover_p26920)
keep_latest_point(hf_recover_p26920)


------------- LF_recover_pt_p26920 -------------
  ⏱️ Latest end_datetime_UTC: 2025-03-24 08:34:43+00:00
  📑 Rows remaining: 1

------------- HF_recover_pt_p26920 -------------
  ⏱️ Latest end_datetime_UTC: 2025-03-24 08:23:44+00:00
  📑 Rows remaining: 1


In [17]:
# ── 3. ADD AND POPULATE 'LABEL' FIELD IN ATTRIBUTE TABLE ────────────────────────────────────────────────

# Function to add 'label' text field to point feature class Attribute Tables and populate it with a given string
def add_label(in_fc, label_text):
    # add 'label' column (data type: text)
    arcpy.management.AddField(
        in_table   = in_fc,
        field_name = "label",
        field_type = "TEXT"
    )
    # populate 'label' column with string ("Deployment Location" or "Recovery Location" based on input feature class)
    with arcpy.da.UpdateCursor(in_fc, ["label"]) as cur:
        for row in cur:
            row[0] = label_text
            cur.updateRow(row)
    print(f"✅ 'Label' field added and populated in {os.path.basename(in_fc)}: '{label_text}'")

# call function for deployment point feature classes using string "Deployment Location"
add_label(lf_deploy_p26920, "Deployment Location")
add_label(hf_deploy_p26920, "Deployment Location")

# call function for recovery point feature classes using string "Recovery Location"
add_label(lf_recover_p26920, "Recovery Location")
add_label(hf_recover_p26920, "Recovery Location")

✅ 'Label' field added and populated in LF_deploy_pt_p26920: 'Deployment Location'
✅ 'Label' field added and populated in HF_deploy_pt_p26920: 'Deployment Location'
✅ 'Label' field added and populated in LF_recover_pt_p26920: 'Recovery Location'
✅ 'Label' field added and populated in HF_recover_pt_p26920: 'Recovery Location'
